In [ ]:
# Celda 1 — Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Librerías cargadas correctamente")

In [ ]:
# Celda 2 — Cargar el archivo de mutaciones
df_raw = pd.read_excel("../Drivers_TF_TVR_TL_to_ALV.xlsx", sheet_name=0, header=None)

print("Archivo cargado")
print(f"Filas: {df_raw.shape[0]}  Columnas: {df_raw.shape[1]}")
print()
print("Estructura de la tabla — cabeceras detectadas (fila 2):")
for i, v in enumerate(df_raw.iloc[2].values[:10]):
    if str(v) != "nan":
        print(f"  Col {i}: {v}")

In [ ]:
# Celda 3 — Extraer y limpiar la tabla de genes ordenada por fusiones
def limpiar_tabla(df_raw, cols_idx, col_names, sort_col=None):
    t = df_raw.iloc[2:, cols_idx].copy()
    t.columns = col_names
    t = t[t["gene"] != "Row Labels"]
    t = t[t["gene"].astype(str) != "Grand Total"]
    t = t.dropna(subset=["gene"])
    t = t[t["gene"].astype(str).str.strip() != ""]
    for col in col_names[1:]:
        t[col] = pd.to_numeric(t[col], errors="coerce")
    t = t.dropna(subset=["avg_fusion"])
    if sort_col:
        t = t.sort_values(sort_col)
    return t.reset_index(drop=True)

# Tabla derecha del Excel: genes ordenados por tasa media de fusiones
genes_ranked = limpiar_tabla(
    df_raw, [5, 6, 7, 8, 9],
    ["gene", "n_pacientes", "avg_fusion", "std_fusion", "orden"],
    sort_col="orden"
)

# Tabla izquierda: todos los genes (para buscar los que no lleguen al filtro n>=10)
genes_todos = limpiar_tabla(
    df_raw, [0, 1, 2, 3, 4],
    ["gene", "n_pacientes", "avg_fusion", "std_fusion", "n2"]
)

# Solo genes con n>=10 pacientes (criterio de significancia del grupo para significancia)
genes_ranked = genes_ranked[genes_ranked["n_pacientes"] >= 10].copy()

print(f"Total genes en el archivo:               {len(genes_todos)}")
print(f"Genes con n >= 10 pacientes:             {len(genes_ranked)}")
print(f"Genes con tasa media >= 0.2 (umbral ALT+): {(genes_ranked['avg_fusion'] >= 0.2).sum()}")
print(f"Tasa media global de fusiones:           0.107")
print()
print("Top 15 genes por tasa media de fusiones:")
print(genes_ranked.head(15).to_string(index=False))

In [ ]:
# Celda 4 — Gráfico: Top 30 genes por tasa de fusiones
ALT_THRESHOLD = 0.2
top30    = genes_ranked.head(30).copy()
genes    = top30["gene"].values
avg_vals = top30["avg_fusion"].values
std_vals = top30["std_fusion"].values
n_vals   = top30["n_pacientes"].values

COLORES = []
for gene, avg in zip(genes, avg_vals):
    if gene == "ATRX":
        COLORES.append("#c0392b")  
    elif gene == "DAXX":
        COLORES.append("#e74c3c")  
    elif avg >= ALT_THRESHOLD:
        COLORES.append("#2c6fad")   
    else:
        COLORES.append("#85b5d9")  

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.bar(range(len(genes)), avg_vals, color=COLORES, alpha=0.85, zorder=3)

sem_vals = std_vals / np.sqrt(n_vals)
ax.errorbar(range(len(genes)), avg_vals, yerr=sem_vals,
            fmt="none", color="#333", capsize=3, linewidth=1, zorder=4)
ax.axhline(y=ALT_THRESHOLD, color="#e67e22", linestyle="--",
           linewidth=1.8, label="Umbral ALT+ (0.2)", zorder=5)

for i, (bar, n, sem) in enumerate(zip(bars, n_vals, sem_vals)):
    ax.text(i, bar.get_height() + float(sem) + 0.01,
            f"n={int(n)}", ha="center", va="bottom", fontsize=6.5, color="#444")

ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Tasa media de fusiones telomericas en tumor", fontsize=11)
ax.set_xlabel("Gen mutado", fontsize=11)
ax.set_title("Top 30 mutaciones driver por tasa media de fusiones\n(genes con n >= 10 pacientes)",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

leyenda = [
    mpatches.Patch(color="#c0392b", label="ATRX — gen ALT canonico principal"),
    mpatches.Patch(color="#e74c3c", label="DAXX — gen ALT canonico (partner)"),
    mpatches.Patch(color="#2c6fad", label="Por encima del umbral ALT+ (>= 0.2)"),
    mpatches.Patch(color="#85b5d9", label="Por debajo del umbral ALT+"),
    plt.Line2D([0],[0], color="#e67e22", linestyle="--", label="Umbral ALT+ (0.2)"),
]
ax.legend(handles=leyenda, loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("img_mutaciones_top30.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafico guardado como img_mutaciones_top30.png")

In [ ]:
# Celda 5 — Gráfico: genes clave del mecanismo ALT
ALT_FOCUS = ["ATRX", "DAXX", "H3-3A", "H3-3B", "POT1", "TERT", "SMARCA4", "CDKN2A", "NF1", "TP53"]

focus_rows = []
for g in ALT_FOCUS:
    row = genes_ranked[genes_ranked["gene"] == g]
    if row.empty:
        row = genes_todos[genes_todos["gene"] == g]
    if not row.empty:
        r = row.iloc[0]
        focus_rows.append({"gene": r["gene"], "n_pacientes": r["n_pacientes"],
                           "avg_fusion": r["avg_fusion"], "std_fusion": r["std_fusion"]})
focus_df = pd.DataFrame(focus_rows)

COLORES2 = ["#c0392b" if g == "ATRX" else "#e74c3c" if g == "DAXX"
            else "#2c6fad" if v >= 0.2 else "#85b5d9"
            for g, v in zip(focus_df["gene"], focus_df["avg_fusion"])]

fig2, ax2 = plt.subplots(figsize=(11, 6))
sem2  = focus_df["std_fusion"] / np.sqrt(focus_df["n_pacientes"])
ax2.bar(range(len(focus_df)), focus_df["avg_fusion"], color=COLORES2, alpha=0.85, zorder=3)
ax2.errorbar(range(len(focus_df)), focus_df["avg_fusion"], yerr=sem2,
             fmt="none", color="#333", capsize=4, linewidth=1.2, zorder=4)
ax2.axhline(y=0.2,   color="#e67e22", linestyle="--", linewidth=1.8, label="Umbral ALT+ (0.2)")
ax2.axhline(y=0.107, color="#aaa",    linestyle=":",  linewidth=1.2, label="Media global (0.107)")

for i, (n, sem, avg) in enumerate(zip(focus_df["n_pacientes"], sem2, focus_df["avg_fusion"])):
    ax2.text(i, float(avg) + float(sem) + 0.005,
             f"n={int(n)}", ha="center", va="bottom", fontsize=9, color="#333")

ax2.set_xticks(range(len(focus_df)))
ax2.set_xticklabels(focus_df["gene"], fontsize=11, fontweight="bold")
ax2.set_ylabel("Tasa media de fusiones telomericas", fontsize=11)
ax2.set_title("Genes clave de mantenimiento de telomeros y actividad ALT",
              fontsize=13, fontweight="bold")
ax2.grid(axis="y", alpha=0.3, zorder=0)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(fontsize=10)
plt.tight_layout()
plt.savefig("img_mutaciones_genes_clave.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafico guardado como img_mutaciones_genes_clave.png")

In [ ]:
# Celda 6 — Tabla resumen: genes con tasa media >= umbral ALT+
genes_alt = genes_ranked[genes_ranked["avg_fusion"] >= 0.2].copy()

print(f"=== GENES CON TASA MEDIA >= 0.2 (umbral ALT+) ===")
print(f"Total: {len(genes_alt)} genes (de {len(genes_ranked)} con n>=10)\n")

print(f"  {'Rank':<6} {'Gen':<14} {'N pacientes':<14} {'Avg fusiones':<14} Nota")
print("-" * 65)
for _, r in genes_alt.iterrows():
    nota = "<-- ALT CANONICO" if r["gene"] in ["ATRX", "DAXX"] else ""
    print(f"  {int(r['orden']):<6} {r['gene']:<14} {int(r['n_pacientes']):<14} {r['avg_fusion']:<14.4f} {nota}")